# PhiUSIIL-A2 → PhreshPhish-A2 zero-shot transfer

Train LightGBM and XGBoost on PhiUSIIL only; freeze model and threshold (0.5); evaluate cleaned PhreshPhish temporal test without retraining. Shared Experiment 5 A2 extractor, exact 18-feature order. Positive class phishing (`y_phish=1`). Five seeds: 13, 42, 2024, 777, 31415. Primary seed 42. PhiUSIIL in-domain A2 test is a separate, non-paired reference. Exploratory: the target set was inspected in prior work.

In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata as md, platform, time
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix
ROOT = Path.cwd().resolve()
while not (ROOT / "Experiment/5 - PhiUSIIL Focused Ablation/src").is_dir():
    if ROOT.parent == ROOT: raise RuntimeError("Run from workspace root or this experiment directory")
    ROOT = ROOT.parent
HERE = ROOT / "Experiment/6 - Cross-Corpus Transfer"
EXP5 = ROOT / "Experiment/5 - PhiUSIIL Focused Ablation"
sys.path.insert(0, str(EXP5))
from src.features import FEATURE_ORDER, url_features, domain_for
from src.data import MODEL_PARAMS, SEEDS, load_phiusiil, load_split, PHRESH_DIR
from src.run_experiment import build_model
assert len(FEATURE_ORDER) == 18 and len(set(FEATURE_ORDER)) == 18
print("workspace", ROOT, "ordered A2 features", FEATURE_ORDER, "seeds", SEEDS)

workspace E:\Kuliah\Tugas Akhir ordered A2 features ['URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'HasObfuscation', 'IsDomainIP'] seeds [13, 42, 2024, 777, 31415]


## Frozen protocol and label verification
PhiUSIIL `train` IDs from frozen random split seed 42 are the only training rows; `val` and `test` do not enter fitting. PhiUSIIL source label is benign=1/phish=0, transformed as `y_phish=1-label`. PhreshPhish source strings map benign→0, phish→1 and are asserted equal to the audited target labels. Target is Experiment 4's cleaned temporal `split_test.csv.gz`; join URLs by immutable `row_id` reconstructed from original test shards and source row position. A2 features are extracted identically from raw URLs, float64, with no learned preprocessing. Model parameters from Experiment 5; four threads; fixed threshold 0.5.

In [2]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1<<20),b""): h.update(block)
    return h.hexdigest()

def extract(urls):
    return pd.DataFrame([url_features(str(u)) for u in urls], columns=FEATURE_ORDER, dtype="float64")

phi_path=ROOT/"Experiment/1 - PhiUSIIL Cleaning + Leakage-Audit Notebook/phiusill_clean/phiusiil_clean.parquet"
split_path=ROOT/"Experiment/1 - PhiUSIIL Cleaning + Leakage-Audit Notebook/phiusill_clean/split_random.csv"
phresh_root=ROOT/"Experiment/4 - PhreshPhish Temporal Audit"
target_path=phresh_root/"outputs/20261022T170935017991Z/split_test.csv.gz"
assert target_path.exists()
# Raw target URLs are used only to reconstruct the already-cleaned target cohort.
parts=[]
for shard in sorted(PHRESH_DIR.glob("test-*.parquet")):
    part=pd.read_parquet(shard,columns=["url","label"])
    part["source_file"]=shard.name
    part["source_row"]=np.arange(len(part))
    parts.append(part)
raw_test=pd.concat(parts,ignore_index=True)
raw_test["row_id"]="test:"+raw_test.source_file+":"+raw_test.source_row.astype(str)
target=pd.read_csv(target_path).merge(raw_test[["row_id","url","label"]],on="row_id",how="left",validate="one_to_one")
assert target.url.notna().all() and target.label.notna().all()
target["label"]=target.label.map({"benign":0,"phish":1}).astype(int)
assert np.array_equal(target.label.to_numpy(),target.y_phish.to_numpy())

phi=load_phiusiil().set_index("url_id")
train_ids,test_ids=load_split("random",42)
Xtr=extract(phi.loc[train_ids.url_id,"URL"])
ytr=phi.loc[train_ids.url_id,"y_phish"].astype(int).to_numpy()
Xref=extract(phi.loc[test_ids.url_id,"URL"])
yref=phi.loc[test_ids.url_id,"y_phish"].astype(int).to_numpy()
Xt=extract(target.url)
yt=target.label.to_numpy()
assert Xtr.columns.tolist()==Xref.columns.tolist()==Xt.columns.tolist()==FEATURE_ORDER
assert set(np.unique(ytr))==set(np.unique(yref))==set(np.unique(yt))=={0,1}
print("train",Xtr.shape,"PhiUSIIL reference",Xref.shape,"PhreshPhish target",Xt.shape,"phishing-positive labels verified")

train (141222, 18) PhiUSIIL reference (47074, 18) PhreshPhish target (168035, 18) phishing-positive labels verified


## Source-only fits and saved predictions
Ten fits total: two model families × five fixed seeds. Each model sees PhiUSIIL training rows only. Threshold and all fitted preprocessing remain fixed for both cohorts.

In [3]:
OUT=HERE/"outputs"
OUT.mkdir(exist_ok=True)
THRESHOLD=0.5
PRIMARY_SEED=42
BOOT_REPS=1000
BOOT_SEED=42

def score(y,p):
    pred=(np.asarray(p)>=THRESHOLD).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"n":len(y),"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
      "f1_phish":f1_score(y,pred,zero_division=0),"precision_phish":precision_score(y,pred,zero_division=0),
      "recall_phish":recall_score(y,pred,zero_division=0),"fpr_benign":fp/(fp+tn) if tn+fp else np.nan,
      "miss_phish":fn/(fn+tp) if fn+tp else np.nan,"roc_auc":roc_auc_score(y,p),
      "average_precision":average_precision_score(y,p),"mcc":matthews_corrcoef(y,pred)}

metrics=[]
prediction_files=[]
for model_name in ("lightgbm","xgboost"):
    for seed in SEEDS:
        model=build_model(model_name,seed,4)
        started=time.perf_counter(); model.fit(Xtr,ytr); fit_seconds=time.perf_counter()-started
        assert np.array_equal(np.asarray(model.classes_),[0,1])
        for cohort,X,y,row_ids,domains in [
            ("phreshphish_temporal_test",Xt,yt,target.row_id.astype(str).to_numpy(),target.domain.astype(str).to_numpy()),
            ("phiusiil_random_test_reference",Xref,yref,test_ids.url_id.to_numpy(),[domain_for(u) for u in phi.loc[test_ids.url_id,"URL"]])]:
            p=model.predict_proba(X)[:,1]
            frame=pd.DataFrame({"cohort":cohort,"model":model_name,"seed":seed,"row_id":row_ids,"domain":domains,"y_phish":y,"p_phish":p,"pred":(p>=THRESHOLD).astype(int)})
            path=OUT/f"predictions_{cohort}_{model_name}_seed{seed}.csv.gz"
            frame.to_csv(path,index=False,compression="gzip"); prediction_files.append(path)
            metrics.append({"cohort":cohort,"model":model_name,"seed":seed,"fit_seconds":fit_seconds,"threshold":THRESHOLD,**score(y,p)})
            print(cohort,model_name,seed,"fit_seconds",round(fit_seconds,2))
metrics_df=pd.DataFrame(metrics)
metrics_path=OUT/"metrics.csv"; metrics_df.to_csv(metrics_path,index=False)
display(metrics_df)

phreshphish_temporal_test lightgbm 13 fit_seconds 2.22


phiusiil_random_test_reference lightgbm 13 fit_seconds 2.22


phreshphish_temporal_test lightgbm 42 fit_seconds 2.03


phiusiil_random_test_reference lightgbm 42 fit_seconds 2.03


phreshphish_temporal_test lightgbm 2024 fit_seconds 2.2


...

phreshphish_temporal_test xgboost 777 fit_seconds 1.79


phiusiil_random_test_reference xgboost 777 fit_seconds 1.79


phreshphish_temporal_test xgboost 31415 fit_seconds 2.19


phiusiil_random_test_reference xgboost 31415 fit_seconds 2.19


,cohort,model,seed,fit_seconds,threshold,n,tn,fp,fn,tp,f1_phish,precision_phish,recall_phish,fpr_benign,miss_phish,roc_auc,average_precision,mcc
0,phreshphish_temporal_test,lightgbm,13,2.217912,0.5,168035,1234,90025,5449,71327,0.599064,0.442058,0.929027,0.986478,0.070973,0.382566,0.400534,-0.146445
1,phiusiil_random_test_reference,lightgbm,13,2.217912,0.5,47074,26942,28,133,19971,0.995985,0.998600,0.993384,0.001038,0.006616,0.998101,0.998554,0.993017
2,phreshphish_temporal_test,lightgbm,42,2.030285,0.5,168035,1234,90025,5449,71327,0.599064,0.442058,0.929027,0.986478,0.070973,0.382566,0.400534,-0.146445
3,phiusiil_random_test_reference,lightgbm,42,2.030285,0.5,47074,26942,28,133,19971,0.995985,0.998600,0.993384,0.001038,0.006616,0.998101,0.998554,0.993017
4,phreshphish_temporal_test,lightgbm,2024,2.203507,0.5,168035,1234,90025,5449,71327,0.599064,0.442058,0.929027,0.986478,0.070973,0.382566,0.400534,-0.146445
5,phiusiil_random_test_reference,lightgbm,2024,2.203507,0.5,47074,26942,28,133,19971,0.995985,0.998600,0.993384,0.001038,0.006616,0.998101,0.998554,0.993017
6,phreshphish_temporal_test,lightgbm,777,2.096036,0.5,168035,1234,90025,5449,71327,0.599064,0.442058,0.929027,0.986478,0.070973,0.382566,0.400534,-0.146445
7,phiusiil_random_test_reference,lightgbm,777,2.096036,0.5,47074,26942,28,133,19971,0.995985,0.998600,0.993384,0.001038,0.006616,0.998101,0.998554,0.993017
8,phreshphish_temporal_test,lightgbm,31415,2.048362,0.5,168035,1234,90025,5449,71327,0.599064,0.442058,0.929027,0.986478,0.070973,0.382566,0.400534,-0.146445
9,phiusiil_random_test_reference,lightgbm,31415,2.048362,0.5,47074,26942,28,133,19971,0.995985,0.998600,0.993384,0.001038,0.006616,0.998101,0.998554,0.993017


## Primary-seed registrable-domain bootstrap
For each model's primary seed (42), sample target registrable domains with replacement 1,000 times (RNG seed 42). Report percentile 95% intervals. This is uncertainty conditional on the one fitted model and one target period, not future-period generalization.

In [4]:
rng=np.random.default_rng(BOOT_SEED)
intervals=[]
for model_name in ("lightgbm","xgboost"):
    frame=pd.read_csv(OUT/f"predictions_phreshphish_temporal_test_{model_name}_seed{PRIMARY_SEED}.csv.gz")
    grouped=frame.groupby("domain",sort=True).indices
    domains=np.array(list(grouped),dtype=object)
    values={k:[] for k in ("f1_phish","precision_phish","recall_phish","fpr_benign","miss_phish","roc_auc","average_precision","mcc")}
    for _ in range(BOOT_REPS):
        chosen=rng.choice(domains,size=len(domains),replace=True)
        ix=np.concatenate([grouped[d] for d in chosen])
        y=frame.y_phish.to_numpy()[ix]; p=frame.p_phish.to_numpy()[ix]
        if len(np.unique(y))<2: continue
        s=score(y,p)
        for key in values: values[key].append(s[key])
    estimate=score(frame.y_phish.to_numpy(),frame.p_phish.to_numpy())
    for key,samples in values.items():
        intervals.append({"cohort":"phreshphish_temporal_test","model":model_name,"seed":PRIMARY_SEED,"metric":key,"estimate":estimate[key],"lower95":np.quantile(samples,.025),"upper95":np.quantile(samples,.975),"valid_replicates":len(samples),"domains":len(domains),"replicates":BOOT_REPS,"bootstrap_seed":BOOT_SEED})
ci=pd.DataFrame(intervals)
ci.to_csv(OUT/"primary_seed_domain_bootstrap.csv",index=False)
assert (ci.valid_replicates==BOOT_REPS).all()
display(ci)

,cohort,model,seed,metric,estimate,lower95,upper95,valid_replicates,domains,replicates,bootstrap_seed
0,phreshphish_temporal_test,lightgbm,42,f1_phish,0.599064,0.538701,0.669829,1000,65288,1000,42
1,phreshphish_temporal_test,lightgbm,42,precision_phish,0.442058,0.381220,0.519504,1000,65288,1000,42
2,phreshphish_temporal_test,lightgbm,42,recall_phish,0.929027,0.914475,0.944563,1000,65288,1000,42
3,phreshphish_temporal_test,lightgbm,42,fpr_benign,0.986478,0.984874,0.988138,1000,65288,1000,42
4,phreshphish_temporal_test,lightgbm,42,miss_phish,0.070973,0.055437,0.085525,1000,65288,1000,42
5,phreshphish_temporal_test,lightgbm,42,roc_auc,0.382566,0.353178,0.413481,1000,65288,1000,42
6,phreshphish_temporal_test,lightgbm,42,average_precision,0.400534,0.357787,0.459478,1000,65288,1000,42
7,phreshphish_temporal_test,lightgbm,42,mcc,-0.146445,-0.175573,-0.111340,1000,65288,1000,42
8,phreshphish_temporal_test,xgboost,42,f1_phish,0.601062,0.541201,0.669748,1000,65288,1000,42
9,phreshphish_temporal_test,xgboost,42,precision_phish,0.444544,0.383774,0.520178,1000,65288,1000,42


## Recompute and provenance manifest
Independently recompute every metric from serialized prediction files (not the in-memory results), then persist the comparison. Hash source inputs and generated results; record package versions and execution environment.

In [5]:
# Independent recomputation is from CSV reloads; compare all metrics and counts.
check_rows=[]
for _,row in metrics_df.iterrows():
    path=OUT/f"predictions_{row.cohort}_{row.model}_seed{int(row.seed)}.csv.gz"
    saved=pd.read_csv(path)
    got=score(saved.y_phish.to_numpy(),saved.p_phish.to_numpy())
    for key,value in got.items():
        expected=row[key]
        assert np.isclose(value,expected,rtol=0,atol=1e-12,equal_nan=True),(path.name,key,value,expected)
        check_rows.append({"file":path.name,"metric":key,"saved_prediction_recompute":value,"metrics_csv":expected,"absolute_difference":abs(value-expected) if np.isfinite(value) else 0.0,"pass":True})
check_path=OUT/"independent_metrics_check.csv"
pd.DataFrame(check_rows).to_csv(check_path,index=False)
input_paths=[phi_path,split_path,target_path,*sorted(PHRESH_DIR.glob("test-*.parquet")),EXP5/"src/features.py",EXP5/"src/data.py",EXP5/"src/run_experiment.py",HERE/"create_notebook.py"]
manifest={"protocol":"A2 zero-shot PhiUSIIL train → cleaned PhreshPhish temporal test","threshold":THRESHOLD,"positive_label":"phishing=1; benign=0","seeds":SEEDS,"primary_seed":PRIMARY_SEED,"feature_order":FEATURE_ORDER,"model_params":MODEL_PARAMS,"threads":4,"bootstrap":{"unit":"registrable domain","replicates":BOOT_REPS,"rng_seed":BOOT_SEED},"input_hashes":{str(p.relative_to(ROOT)):sha256(p) for p in input_paths},"output_hashes":{p.name:sha256(p) for p in [metrics_path,OUT/"primary_seed_domain_bootstrap.csv",check_path,*prediction_files]},"environment":{"python":platform.python_version(),"packages":{name:md.version(name) for name in ["numpy","pandas","scikit-learn","lightgbm","xgboost","pyarrow","tldextract","nbformat","nbclient"]}},"rows":{"phiusiil_train":len(Xtr),"phiusiil_reference_test":len(Xref),"phreshphish_temporal_test":len(Xt)},"independent_prediction_metric_checks":len(check_rows)}
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(json.dumps(manifest,indent=2))

{
  "protocol": "A2 zero-shot PhiUSIIL train \u2192 cleaned PhreshPhish temporal test",
  "threshold": 0.5,
  "positive_label": "phishing=1; benign=0",
  "seeds": [
    13,
    42,
    2024,
    777,
    31415
  ],
  "primary_seed": 42,
  "feature_order": [
    "URLLength",
    "DomainLength",
    "TLDLength",
    "NoOfSubDomain",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "NoOfLettersInURL",
    "LetterRatioInURL",
    "NoOfDegitsInURL",
    "DegitRatioInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL",
    "IsHTTPS",
    "HasObfuscation",
    "IsDomainIP"
  ],
  "model_params": {
    "lightgbm": {
      "n_estimators": 300,
      "learning_rate": 0.05,
      "num_leaves": 63,
      "verbose": -1
    },
    "xgboost": {
      "n_estimators": 300,
      "learning_rate": 0.05,
      "max_depth": 8,
      "eval_metric": "logloss",
      "tree_method": "hist"
    }
  },
  "threads": 4,
  "

## Limitations
- No PhreshPhish target data used during fitting or threshold selection.
- PhiUSIIL reference is in-domain and non-paired; cross-dataset difference is not a controlled causal contrast.
- Earlier target inspection makes this exploratory, not blind prospective testing.
- Five seeds measure training stochasticity, not independent target periods.
- Target period reflects time and population shift together.
- Registrable domains can combine unrelated tenants; bootstrap does not establish independent organizations. Private PSL suffixes disabled by the shared extractor.
- Dataset collection and label provenance may differ. Results apply only to the documented cleaned target cohort.